# QUONITRON overlap repair lab (Parts 1–4 only)
Select the VS Code kernel that has this project's render dependencies. Run all cells.
This uses ordinary `Scene.render()`, `tempconfig`, and `IPython.display.Video`, not an assumed Manim notebook magic. A missing renderer is reported as **blocked**, never as a visual pass.

Install the native Cairo/Pango/FFmpeg dependencies following README first, then in that kernel run `%pip install -e ".[render,dev]"`. EquationText may use its existing plain-text fallback when TeX is missing. No package installation is performed by this notebook.

In [1]:
import sys, shutil, importlib.util
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/quonitron_manim").is_dir(), "Open this notebook from the repository or notebooks directory"
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from quonitron_manim.core.manim_compat import cairo_usable, manimpango_usable, manim_available
print("Kernel:", sys.executable)
print("Cairo:", cairo_usable())
print("Pango:", manimpango_usable())
print("FFmpeg:", shutil.which("ffmpeg"))
READY = manim_available() and cairo_usable()[0] and manimpango_usable()[0] and bool(shutil.which("ffmpeg"))
print("Render status:", "READY (not yet rendered)" if READY else "BLOCKED: no visual verification")

Kernel: /home/user/manim-explainer-/.venv/bin/python
Cairo: (False, "pycairo import failed: No module named 'cairo'")
Pango: (False, "manimpango import failed: No module named 'manimpango'")
FFmpeg: None
Render status: BLOCKED: no visual verification


## Layout warning example (works without Manim)
This deliberately overlaps two unrelated semantic objects and reports IDs and scene name. Expected panel containment is then explicitly accepted. No implicit containment exception hides accidental collisions.

In [2]:
from quonitron_manim.design.layout import LayoutValidator, LayoutItem
class DemoBox:
    def get_left(self): return [-1, 0, 0]
    def get_right(self): return [1, 0, 0]
    def get_bottom(self): return [0, 0, 0]
    def get_top(self): return [0, 1, 0]
validator = LayoutValidator(scene="NotebookCollisionExample")
items = [LayoutItem("panel", DemoBox()), LayoutItem("explanation", DemoBox())]
for issue in validator.validate(items): print(issue)
validator.allow("panel", "explanation", reason="text deliberately contained in panel")
for issue in validator.validate(items): print(issue)

ERROR [NotebookCollisionExample] panel, explanation: collision (2.000 x 1.000); reposition or separate beats
INFO [NotebookCollisionExample] panel, explanation: intentional overlap: text deliberately contained in panel


## Render helper
`Video(embed=True)` embeds the generated MP4 where the VS Code frontend supports it. If inline video is unsupported, open the printed local MP4 path. The render output is not a visual approval; inspect the animation yourself.

In [3]:
def render_demo(filename, scene_name):
    if not READY:
        print(f"SKIPPED {scene_name}: renderer dependencies unavailable")
        return None
    from manim import tempconfig
    from IPython.display import Video, display
    spec = importlib.util.spec_from_file_location("notebook_scene", ROOT / "examples" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    with tempconfig({"quality": "low_quality", "media_dir": str(ROOT / "media/layout-notebook"),
                     "disable_caching": True, "preview": False}):
        scene = getattr(module, scene_name)()
        scene.render()
        movie = Path(scene.renderer.file_writer.movie_file_path)
    print(movie)
    display(Video(filename=str(movie), embed=True))
    return movie

## Long explanation, diagram labels/arrows, subtitle band, and transition
The repair exercise keeps fixed-size text, measures wrapping using Pango geometry, tries label positions against the diagram, and swaps unrelated paragraphs sequentially. At the end it prints a deliberate collision diagnostic. Inspect the transition as well as settled frames.

In [4]:
exercise_movie = render_demo("layout_audit_demo.py", "LayoutRepairExercise")

SKIPPED LayoutRepairExercise: renderer dependencies unavailable


## Repaired existing Part 4
Inspect approximately 28 s (annotation), 38 s (counter replacement), 43 s (steps), 48 s (recap), and 53 s (long paragraph). The steps must disappear before the recap enters; the recap must disappear before the paragraph enters.

In [5]:
text_movie = render_demo("text_typography_demo.py", "AssignmentInType")

SKIPPED AssignmentInType: renderer dependencies unavailable


## Existing diagram / camera transition
The subtitle is projected into the moving camera viewport on every update. Close-up cropping of the surrounding table/heading is intentional focus, but runtime bounds diagnostics are retained rather than suppressed. Inspect captions at 49 s and camera restoration at 54 s.

In [6]:
journey_movie = render_demo("journey_demo.py", "CodeToMemoryJourney")

SKIPPED CodeToMemoryJourney: renderer dependencies unavailable


## Long-text overflow is actionable, not silently shrunk
An excessively long cue must be split in the timeline. This example intentionally catches the failure.

In [7]:
if READY:
    from quonitron_manim.core.text import subtitle_text
    from quonitron_manim.design.layout import LayoutError
    try:
        subtitle_text("This explanation needs multiple timed cues. " * 80)
    except LayoutError as error:
        print("Expected overflow:", error)
else:
    print("SKIPPED measured Pango long-text test: renderer unavailable")

SKIPPED measured Pango long-text test: renderer unavailable


## Extract review frames for all four existing demos
Optional: run `python scripts/render_layout_audit.py` in the same environment. It renders the demos and extracts timestamped PNGs with FFmpeg, including transition samples. These are review artifacts, **not pixel-regression baselines**. No visually verified baseline was available in the repair sandbox.